# Graph RAG + Qdrant POC — Microsoft GraphRAG Edition

**Architecture:**
```
JSON Records (Uber marketplace — drivers, riders, trips, reviews, zones)
       │
       ▼  Section 4
Convert JSON to plain-text input files for graphrag
       │
       ▼  Section 5
Write settings.yaml (completion_models / embedding_models format)
and .env  ->  graphrag v2 project ready
       │
       ▼  Section 6
graphrag index  ->  LLM extracts entities, relationships, communities,
                    generates community summaries
                    (uses LanceDB INTERNALLY as its own vector store)
                    ->  parquet files written to output/
       │
       ▼  Section 7
Load parquet outputs with graphrag indexer adapters:
  read_indexer_entities / read_indexer_communities /
  read_indexer_reports  / read_indexer_relationships /
  read_indexer_text_units
       │
       ├── Section 8:  Dense embeddings (OpenAI text-embedding-3-small)
       └── Section 9:  Sparse vectors  (BM25 with rebuild_bm25 for new docs)
              │
              ▼  Section 10
       Qdrant Serverless -- named vectors: 'dense' + 'sparse'
       Two collections: uber_entities  and  uber_communities
              │
       Section 11: CRUD (Create/Read/Update/Delete) on Qdrant
       Section 12: Dense search, Sparse search, Hybrid RRF search
       Section 13: Official graphrag.api local_search + Qdrant hybrid local
       Section 14: Official graphrag.api global_search + Qdrant hybrid global
       Section 15: Live ingestion demo (new driver -> re-index -> upsert)
       Section 16: Search comparison table
       Section 17: Summary
```

**Key technology:** Microsoft GraphRAG (`graphrag>=2.0.0`) for knowledge graph construction, Qdrant for hybrid vector search.

> **Two-layer vector store design — important distinction:**
>
> GraphRAG uses **LanceDB internally** as its own vector store during the indexing pipeline. This is baked into graphrag and we do NOT replace or configure it. After indexing completes, we read the **parquet outputs** (entities, communities, community_reports, relationships, text_units) and ingest them into **Qdrant as an additional, independent hybrid-search layer**. The Qdrant layer adds BM25 sparse search, RRF fusion, CRUD operations, and payload filtering on top of the graphrag knowledge graph.
>
> We also demonstrate the **official `graphrag.api`** Python interface for local and global search, which works directly from the parquet outputs via `load_config`. Both search paths (graphrag native API and Qdrant hybrid) are shown side-by-side.
>
> Reference: https://qdrant.tech/documentation/frameworks/microsoft-graphrag/

## Section 1 — Install Dependencies

In [ ]:
%pip install -q "graphrag>=2.0.0" "qdrant-client>=1.10.0" openai pandas numpy rank-bm25 tqdm python-dotenv

## Section 2 — Configuration (Colab Secrets OR .env)

In [ ]:
import os

try:
    from google.colab import userdata
    OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
    QDRANT_URL     = userdata.get("QDRANT_URL")
    QDRANT_API_KEY = userdata.get("QDRANT_API_KEY")
    print("Loaded from Colab Secrets.")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()
    OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
    QDRANT_URL     = os.getenv("QDRANT_URL")
    QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")
    print("Loaded from .env")

missing = [k for k, v in {
    "OPENAI_API_KEY": OPENAI_API_KEY,
    "QDRANT_URL": QDRANT_URL,
    "QDRANT_API_KEY": QDRANT_API_KEY
}.items() if not v]

if missing:
    raise EnvironmentError(f"Missing credentials: {missing}")

print(f"OpenAI key: ...{OPENAI_API_KEY[-4:]}")
print(f"Qdrant URL: {QDRANT_URL}")
print(f"Qdrant key: ...{QDRANT_API_KEY[-4:]}")

## Section 3 — Source Data (Uber Marketplace JSON)

In [ ]:
# Uber-like marketplace data

DRIVERS = [
    {"id": "D001", "name": "Carlos Mendez",  "vehicle": "Toyota Camry 2021",  "rating": 4.92,
     "trips": 1843, "zone": "Downtown",   "license": "CM-4821", "years_active": 5,
     "languages": ["English", "Spanish"], "specialty": "Airport transfers"},
    {"id": "D002", "name": "Priya Sharma",   "vehicle": "Honda Accord 2022",  "rating": 4.87,
     "trips": 1204, "zone": "Midtown",    "license": "PS-3317", "years_active": 3,
     "languages": ["English", "Hindi"],   "specialty": "Corporate rides"},
    {"id": "D003", "name": "James Okafor",   "vehicle": "Ford Fusion 2020",   "rating": 4.78,
     "trips": 987,  "zone": "Airport",    "license": "JO-7762", "years_active": 2,
     "languages": ["English", "Yoruba"],  "specialty": "Night rides"},
    {"id": "D004", "name": "Mei Lin",        "vehicle": "Tesla Model 3 2023", "rating": 4.95,
     "trips": 2341, "zone": "Suburbs",    "license": "ML-9901", "years_active": 6,
     "languages": ["English", "Mandarin"],"specialty": "Luxury rides"},
    {"id": "D005", "name": "Ahmed Hassan",   "vehicle": "Hyundai Sonata 2021","rating": 4.83,
     "trips": 1567, "zone": "Downtown",   "license": "AH-5543", "years_active": 4,
     "languages": ["English", "Arabic"],  "specialty": "Family rides"}
]

RIDERS = [
    {"id": "R001", "name": "Sarah Johnson",  "tier": "Gold",     "trips": 312, "home_zone": "Downtown",
     "preferred_vehicle": "Sedan",  "rating": 4.9, "frequent_route": "Downtown to Airport"},
    {"id": "R002", "name": "Tom Bradley",    "tier": "Silver",   "trips": 87,  "home_zone": "Midtown",
     "preferred_vehicle": "SUV",    "rating": 4.6, "frequent_route": "Midtown to Suburbs"},
    {"id": "R003", "name": "Lisa Chen",      "tier": "Platinum", "trips": 678, "home_zone": "Suburbs",
     "preferred_vehicle": "Tesla",  "rating": 5.0, "frequent_route": "Suburbs to Downtown"},
    {"id": "R004", "name": "Marcus Williams","tier": "Silver",   "trips": 156, "home_zone": "Airport",
     "preferred_vehicle": "Sedan",  "rating": 4.7, "frequent_route": "Airport to Midtown"},
    {"id": "R005", "name": "Emma Davis",     "tier": "Bronze",   "trips": 23,  "home_zone": "Midtown",
     "preferred_vehicle": "Any",    "rating": 4.4, "frequent_route": "Midtown to Downtown"}
]

TRIPS = [
    {"id": "T001", "driver": "D001", "rider": "R001", "from": "Downtown",  "to": "Airport",
     "fare": 34.50, "duration_min": 28, "date": "2024-01-15", "surge": 1.2, "status": "completed"},
    {"id": "T002", "driver": "D002", "rider": "R002", "from": "Midtown",   "to": "Suburbs",
     "fare": 22.75, "duration_min": 35, "date": "2024-01-16", "surge": 1.0, "status": "completed"},
    {"id": "T003", "driver": "D004", "rider": "R003", "from": "Suburbs",   "to": "Downtown",
     "fare": 41.00, "duration_min": 42, "date": "2024-01-17", "surge": 1.5, "status": "completed"},
    {"id": "T004", "driver": "D003", "rider": "R004", "from": "Airport",   "to": "Midtown",
     "fare": 29.80, "duration_min": 33, "date": "2024-01-18", "surge": 1.0, "status": "completed"},
    {"id": "T005", "driver": "D005", "rider": "R005", "from": "Midtown",   "to": "Downtown",
     "fare": 15.20, "duration_min": 18, "date": "2024-01-19", "surge": 1.0, "status": "completed"},
    {"id": "T006", "driver": "D001", "rider": "R003", "from": "Downtown",  "to": "Suburbs",
     "fare": 38.90, "duration_min": 45, "date": "2024-01-20", "surge": 1.3, "status": "completed"}
]

REVIEWS = [
    {"id": "REV001", "trip": "T001", "driver": "D001", "rider": "R001", "stars": 5,
     "text": "Carlos was punctual, professional, and knew the fastest route to the airport. Highly recommend!"},
    {"id": "REV002", "trip": "T002", "driver": "D002", "rider": "R002", "stars": 4,
     "text": "Priya's car was clean and she was very polite. Traffic was bad but she handled it well."},
    {"id": "REV003", "trip": "T003", "driver": "D004", "rider": "R003", "stars": 5,
     "text": "Mei Lin's Tesla was amazing -- silent, smooth, and arrived two minutes early. Perfect 5 stars."},
    {"id": "REV004", "trip": "T004", "driver": "D003", "rider": "R004", "stars": 4,
     "text": "James knew the airport pickup zone well. Slightly early arrival but accommodating."},
    {"id": "REV005", "trip": "T005", "driver": "D005", "rider": "R005", "stars": 5,
     "text": "Ahmed was super friendly and made the short ride enjoyable. Great driver!"},
    {"id": "REV006", "trip": "T006", "driver": "D001", "rider": "R003", "stars": 5,
     "text": "Carlos again! Consistently the best driver in the app. Smooth ride, great conversation."},
    {"id": "REV007", "trip": "T002", "driver": "D002", "rider": "R002", "stars": 4,
     "text": "Priya is always reliable for the Midtown to Suburbs route. Regular go-to driver."}
]

ZONES = [
    {"id": "Z001", "name": "Downtown",  "demand_score": 9.2, "avg_wait_min": 3.1,
     "surge_freq": "high",   "peak_hours": "7-9am, 5-8pm",  "top_destinations": ["Airport", "Midtown"]},
    {"id": "Z002", "name": "Midtown",   "demand_score": 7.8, "avg_wait_min": 4.5,
     "surge_freq": "medium", "peak_hours": "8-10am, 6-9pm", "top_destinations": ["Downtown", "Suburbs"]},
    {"id": "Z003", "name": "Airport",   "demand_score": 8.5, "avg_wait_min": 2.8,
     "surge_freq": "high",   "peak_hours": "6-8am, 4-7pm",  "top_destinations": ["Downtown", "Midtown"]},
    {"id": "Z004", "name": "Suburbs",   "demand_score": 5.3, "avg_wait_min": 7.2,
     "surge_freq": "low",    "peak_hours": "7-9am, 5-7pm",  "top_destinations": ["Downtown", "Midtown"]}
]

print(f"Dataset: {len(DRIVERS)} drivers, {len(RIDERS)} riders, {len(TRIPS)} trips, {len(REVIEWS)} reviews, {len(ZONES)} zones")

## Section 4 — Prepare GraphRAG Input Files

Convert JSON records to rich natural-language text documents (200-400 words each) that Microsoft GraphRAG can process for entity and relationship extraction.

In [ ]:
import os

GRAPHRAG_ROOT = "./graphrag_project"
INPUT_DIR = f"{GRAPHRAG_ROOT}/input"
os.makedirs(INPUT_DIR, exist_ok=True)

# File 1: Drivers and Vehicles
drivers_text = """# Uber Marketplace Driver Profiles and Vehicle Information

This document describes the professional drivers operating within the Uber marketplace platform,
their vehicles, service areas, and expertise.

## Driver: Carlos Mendez (ID: D001)
Carlos Mendez is a highly experienced driver based in the Downtown zone with 5 years of active
service on the Uber platform. He operates a Toyota Camry 2021 (license CM-4821) and has
completed an impressive 1,843 trips, earning a stellar 4.92 out of 5 rating. Carlos specializes
in airport transfers and is fluent in both English and Spanish, making him exceptionally popular
with international travelers. His knowledge of the fastest routes between Downtown and the Airport
has made him one of the most sought-after drivers on the platform. Riders consistently praise
Carlos for his punctuality, professionalism, and route expertise. He has served riders like
Sarah Johnson and Lisa Chen on multiple occasions, earning repeat business from loyal customers.

## Driver: Priya Sharma (ID: D002)
Priya Sharma operates out of the Midtown zone and has been driving for the Uber platform for
3 years. She drives a Honda Accord 2022 (license PS-3317) and has completed 1,204 trips with
a 4.87 rating. Priya specializes in corporate rides and is bilingual in English and Hindi.
Her pristine vehicle and professional demeanor make her the preferred choice for business
travelers needing rides between Midtown and Suburbs. Priya is known for maintaining an
immaculate, clean vehicle and her calm handling of heavy traffic situations. Tom Bradley is
among her regular riders who consistently choose her for the Midtown to Suburbs corridor.

## Driver: James Okafor (ID: D003)
James Okafor is a 2-year veteran driver stationed at the Airport zone. He drives a Ford Fusion
2020 (license JO-7762) and has accumulated 987 completed trips with a 4.78 rating. James
specializes in night rides and late-night airport pickups, speaking both English and Yoruba.
His deep familiarity with airport pickup and drop-off procedures, terminal locations, and
optimal routing between the Airport and Midtown zones makes him invaluable for travelers.
James is recognized for his accommodating nature and punctuality at the airport zone, serving
riders like Marcus Williams who regularly travel between the Airport and Midtown.

## Driver: Mei Lin (ID: D004)
Mei Lin is the platform's top-rated driver with a near-perfect 4.95 rating and the highest
trip count of 2,341 completed rides over 6 years. She operates a Tesla Model 3 2023
(license ML-9901) from the Suburbs zone, specializing in luxury rides. Fluent in English and
Mandarin, Mei Lin attracts premium passengers seeking a quiet, smooth, eco-friendly experience.
Her electric vehicle offers a distinctive silent ride that riders consistently highlight in
five-star reviews. The Suburbs to Downtown route is her primary corridor, where she provides
an unmatched luxury transportation experience. Lisa Chen, a Platinum-tier rider, is among
her most loyal customers.

## Driver: Ahmed Hassan (ID: D005)
Ahmed Hassan has been driving in the Downtown zone for 4 years, completing 1,567 trips with
a solid 4.83 rating. He drives a Hyundai Sonata 2021 (license AH-5543) and specializes in
family rides. Speaking English and Arabic, Ahmed serves a diverse clientele. His friendly
personality and ability to accommodate families with children and luggage have earned him a
loyal customer base. Ahmed is particularly popular for short-distance rides within and around
the Downtown zone, often serving newer riders like Emma Davis.
"""

# File 2: Trips and Reviews
trips_reviews_text = """# Uber Trip Records and Rider Reviews

This document captures completed trip records along with authentic rider reviews from the
Uber marketplace platform, documenting the service quality and rider experiences.

## Trip T001: Downtown to Airport -- January 15, 2024
Driver Carlos Mendez (D001) transported rider Sarah Johnson (R001) from Downtown to the
Airport on January 15, 2024. The trip lasted 28 minutes with a fare of $34.50, including
a 1.2x surge multiplier due to morning rush hour demand. Sarah, a Gold-tier rider who
frequently travels this exact route, rated the experience 5 stars.
Review: "Carlos was punctual, professional, and knew the fastest route to the airport.
Highly recommend!"

## Trip T002: Midtown to Suburbs -- January 16, 2024
Driver Priya Sharma (D002) completed a ride for Tom Bradley (R002) from Midtown to Suburbs
on January 16, 2024. The 35-minute journey cost $22.75 with no surge pricing. Tom, a
Silver-tier rider, gave Priya a 4-star rating.
Review: "Priya's car was clean and she was very polite. Traffic was bad but she handled
it well." Tom later added another review: "Priya is always reliable for the Midtown to
Suburbs route. Regular go-to driver."

## Trip T003: Suburbs to Downtown -- January 17, 2024
Mei Lin (D004) drove Platinum-tier rider Lisa Chen (R003) from Suburbs to Downtown on
January 17, 2024. This 42-minute premium ride cost $41.00 with a 1.5x surge multiplier
during peak hours. Lisa, who has taken 678 trips and maintains a perfect 5.0 rider rating,
left a glowing 5-star review.
Review: "Mei Lin's Tesla was amazing -- silent, smooth, and arrived two minutes early.
Perfect 5 stars."

## Trip T004: Airport to Midtown -- January 18, 2024
James Okafor (D003) picked up Marcus Williams (R004) from the Airport and drove him to
Midtown on January 18, 2024. The trip took 33 minutes and cost $29.80 with no surge.
Marcus, a Silver-tier rider who frequently travels between Airport and Midtown, rated the
trip 4 stars.
Review: "James knew the airport pickup zone well. Slightly early arrival but accommodating."

## Trip T005: Midtown to Downtown -- January 19, 2024
Ahmed Hassan (D005) completed a short 18-minute ride for Emma Davis (R005) from Midtown
to Downtown on January 19, 2024, costing just $15.20 with no surge. Emma, a new Bronze-tier
rider with only 23 trips, gave Ahmed a perfect 5-star rating.
Review: "Ahmed was super friendly and made the short ride enjoyable. Great driver!"

## Trip T006: Downtown to Suburbs -- January 20, 2024
Carlos Mendez (D001) drove Lisa Chen (R003) from Downtown to Suburbs on January 20, 2024.
This was the second time Lisa rode with Carlos, demonstrating his cross-zone flexibility.
The 45-minute trip cost $38.90 with a 1.3x surge multiplier. Lisa again rated the experience
5 stars.
Review: "Carlos again! Consistently the best driver in the app. Smooth ride, great
conversation."

## Service Quality Summary
Across all recorded trips, the platform maintains an average driver rating of 4.87 stars.
Carlos Mendez stands out with multiple 5-star reviews from different riders including both
Sarah Johnson and Lisa Chen. Mei Lin's electric Tesla vehicle consistently receives praise
for the unique luxury riding experience it provides. The Downtown and Airport zones generate
the most high-value trips with surge pricing due to high demand. Priya Sharma has built
a loyal following among corporate travelers in the Midtown to Suburbs corridor.
"""

# File 3: Zones and Riders
zones_riders_text = """# Uber Service Zones and Rider Profiles

This document describes the geographic service zones in the Uber marketplace and the
rider community using the platform.

## Service Zone: Downtown (Z001)
The Downtown zone is the platform's highest-demand area with a demand score of 9.2 out of 10.
Riders in Downtown experience an average wait time of just 3.1 minutes, reflecting the dense
concentration of available drivers. Surge pricing occurs frequently in Downtown, particularly
during peak hours of 7-9am and 5-8pm. The most popular destinations from Downtown are the
Airport and Midtown. Two of the platform's most experienced drivers, Carlos Mendez and
Ahmed Hassan, are based primarily in the Downtown zone, ensuring high service availability.
The Downtown zone serves a mix of business commuters and leisure travelers.

## Service Zone: Midtown (Z002)
Midtown operates with a demand score of 7.8, representing moderate-to-high ride volume.
The average wait time is 4.5 minutes, and surge pricing occurs at medium frequency during
peak hours of 8-10am and 6-9pm. Riders most commonly travel from Midtown to Downtown or
Suburbs. Priya Sharma is the primary driver serving the Midtown zone, particularly popular
with corporate clients who require reliable, professional transportation. The zone has a
high concentration of office buildings and corporate headquarters.

## Service Zone: Airport (Z003)
The Airport zone has a high demand score of 8.5 and the shortest average wait time of only
2.8 minutes, as dedicated airport drivers maintain constant availability for arriving
passengers. Surge pricing is common during flight arrival peaks at 6-8am and 4-7pm.
The Airport zone primarily sends riders to Downtown and Midtown. James Okafor specializes
in this zone, leveraging his knowledge of terminal layouts and pickup procedures. The Airport
zone is critical infrastructure for business travelers and tourists entering the city.

## Service Zone: Suburbs (Z004)
The Suburbs zone has the lowest demand score of 5.3 and the longest average wait time of
7.2 minutes. Surge pricing is rare in this zone. Despite lower overall demand, the Suburbs
zone generates high-value individual trips as riders travel long distances to Downtown or
Midtown. Mei Lin's premium Tesla service from the Suburbs has elevated the zone's reputation
for luxury transportation, attracting affluent residents who prefer eco-conscious premium rides.

## Rider Profile: Sarah Johnson (R001)
Sarah Johnson is a Gold-tier rider with 312 completed trips, reflecting regular platform usage.
She maintains a high rider rating of 4.9 and lives in the Downtown zone. Sarah prefers sedan
vehicles and most frequently travels the Downtown to Airport route, suggesting business travel
or regular commuting. Her loyalty to the platform and high trip count make her a valuable
Gold-tier member. Carlos Mendez is her preferred driver for airport transfers.

## Rider Profile: Tom Bradley (R002)
Tom Bradley holds Silver-tier status with 87 trips from his Midtown home zone. He prefers
SUV vehicles and regularly travels between Midtown and Suburbs. His 4.6 rider rating reflects
generally positive interactions with drivers. Priya Sharma is his most frequently used driver
for the Midtown to Suburbs commute route.

## Rider Profile: Lisa Chen (R003)
Lisa Chen is a Platinum-tier rider, the highest loyalty tier, with an impressive 678 trips
and a perfect 5.0 rider rating. Based in Suburbs, she prefers Tesla vehicles and frequently
travels to Downtown. Lisa has ridden with both Mei Lin and Carlos Mendez, and her reviews
consistently award 5 stars, making her one of the platform's most valuable and appreciated
customers. Her preference for premium electric vehicles reflects an eco-conscious lifestyle.

## Rider Profile: Marcus Williams (R004)
Marcus Williams is a Silver-tier rider based at the Airport zone who has completed 156 trips.
He prefers sedans and regularly travels between the Airport and Midtown. His 4.7 rider rating
indicates positive service interactions. James Okafor is his go-to driver for airport connections.

## Rider Profile: Emma Davis (R005)
Emma Davis is a new Bronze-tier rider with only 23 trips completed from her Midtown home zone.
She is flexible on vehicle type and primarily travels to Downtown. Emma's 4.4 rider rating and
enthusiastic reviews suggest she is developing a positive relationship with the platform.
Ahmed Hassan provided her first memorable ride experience in the platform.
"""

with open(f"{INPUT_DIR}/drivers_and_vehicles.txt", "w") as f:
    f.write(drivers_text)

with open(f"{INPUT_DIR}/trips_and_reviews.txt", "w") as f:
    f.write(trips_reviews_text)

with open(f"{INPUT_DIR}/zones_and_riders.txt", "w") as f:
    f.write(zones_riders_text)

for fname in os.listdir(INPUT_DIR):
    path = f"{INPUT_DIR}/{fname}"
    size = os.path.getsize(path)
    print(f"  {fname}: {size} bytes")

print("\nGraphRAG input files created successfully.")

## Section 5 — Initialize GraphRAG Project

Write `settings.yaml` and `.env` manually using the **GraphRAG v2 API format**.

Key differences from older versions:
- Top-level keys are `completion_models` and `embedding_models` (NOT `models`)
- Model config uses `model_provider` + `auth_method` (NOT `type: openai_chat`)
- Entity extraction is under `extract_graph:` (NOT `entity_extraction:`)

In [ ]:
import os

GRAPHRAG_ROOT = "./graphrag_project"

# GraphRAG v2 settings.yaml format:
#   - completion_models / embedding_models  (NOT 'models')
#   - model_provider + auth_method          (NOT 'type: openai_chat')
#   - extract_graph                         (NOT 'entity_extraction')
settings_yaml = """
completion_models:
  default_completion_model:
    model_provider: openai
    model: gpt-4o-mini
    auth_method: api_key
    api_key: ${GRAPHRAG_API_KEY}
    max_tokens: 4000

embedding_models:
  default_embedding_model:
    model_provider: openai
    model: text-embedding-3-small
    auth_method: api_key
    api_key: ${GRAPHRAG_API_KEY}

input:
  type: file
  file_type: text
  base_dir: "input"

output:
  type: file
  base_dir: "output"

chunks:
  size: 800
  overlap: 100

extract_graph:
  entity_types: [driver, rider, zone, vehicle, trip]
  max_gleanings: 1

community_reports:
  max_length: 1500
  max_input_length: 8000

claim_extraction:
  enabled: false
"""

with open(f"{GRAPHRAG_ROOT}/settings.yaml", "w") as f:
    f.write(settings_yaml)

with open(f"{GRAPHRAG_ROOT}/.env", "w") as f:
    f.write(f"GRAPHRAG_API_KEY={OPENAI_API_KEY}\n")

print("GraphRAG v2 project initialized.")
print(f"  settings.yaml : {os.path.getsize(GRAPHRAG_ROOT+'/settings.yaml')} bytes")
print(f"  .env          : created (API key stored as GRAPHRAG_API_KEY)")
print(f"  input/ files  : {len(os.listdir(GRAPHRAG_ROOT+'/input'))} documents")

## Section 6 — Run GraphRAG Index

> **Note:** This cell calls the OpenAI API approximately 15-30 times to extract entities, build the knowledge graph, detect communities, and generate community summaries.  
> **Estimated cost:** < $0.10 with `gpt-4o-mini`  
> **Estimated runtime:** 2-5 minutes depending on API response times.

In [ ]:
import subprocess

print("Starting graphrag indexing pipeline...")
print("This will call OpenAI APIs ~15-30 times. Expected runtime: 2-5 minutes.\n")

result = subprocess.run(
    ["python", "-m", "graphrag", "index", "--root", GRAPHRAG_ROOT],
    capture_output=True,
    text=True
)

stdout = result.stdout
print(stdout[-3000:] if len(stdout) > 3000 else stdout)

if result.returncode != 0:
    print("\n--- STDERR ---")
    print(result.stderr[-2000:])
    print(f"\nReturn code: {result.returncode}")
else:
    print("\nGraphRAG indexing completed successfully!")

## Section 7 — Load GraphRAG Parquet Outputs

Use the **official `graphrag` indexer adapters** to load the parquet files produced by Section 6. These adapter functions handle schema normalization so the output objects are compatible with `graphrag.api` local/global search.

Also load the graphrag config object via `load_config` — required for the native `graphrag.api` search calls in Section 13/14.

> **Note on LanceDB:** GraphRAG wrote an internal LanceDB vector store into `output/` during indexing. We do NOT touch that. We read only the parquet files and feed them into Qdrant (our own separate layer) and into `graphrag.api` (native search).

In [ ]:
import pandas as pd
import glob
import os

output_dir = f"{GRAPHRAG_ROOT}/output"

def load_parquet(name):
    """Search recursively for a parquet file by stem name."""
    files = glob.glob(f"{output_dir}/**/{name}.parquet", recursive=True)
    if not files:
        files = glob.glob(f"{output_dir}/**/*{name}*.parquet", recursive=True)
    if not files:
        print(f"  WARNING: {name}.parquet not found in {output_dir}")
        all_parquets = glob.glob(f"{output_dir}/**/*.parquet", recursive=True)
        if all_parquets:
            print(f"  Available parquet files:")
            for p in all_parquets:
                print(f"    {p}")
        else:
            print(f"  No parquet files found at all. Has Section 6 been run?")
        return pd.DataFrame()
    df = pd.read_parquet(files[0])
    print(f"  Loaded {name}.parquet from {files[0]}  ({len(df)} rows, cols: {list(df.columns)[:6]})")
    return df

print("Loading GraphRAG outputs...")
try:
    entities_df          = load_parquet("entities")
    relationships_df     = load_parquet("relationships")
    community_reports_df = load_parquet("community_reports")
    text_units_df        = load_parquet("text_units")
except Exception as e:
    print(f"Error loading parquet files: {e}")
    print("Initializing empty DataFrames -- Section 8 will use fallback source data.")
    entities_df = relationships_df = community_reports_df = text_units_df = pd.DataFrame()

print(f"\nSummary:")
print(f"  entities          : {len(entities_df)} rows")
print(f"  relationships     : {len(relationships_df)} rows")
print(f"  community_reports : {len(community_reports_df)} rows")
print(f"  text_units        : {len(text_units_df)} rows")

In [ ]:
# Show sample entities
if not entities_df.empty:
    print("=== Sample Entities (first 5) ===")
    cols = [c for c in ["title", "type", "description", "rank"] if c in entities_df.columns]
    pd.set_option("display.max_colwidth", 80)
    print(entities_df[cols].head())
else:
    print("entities_df is empty -- run Section 6 first to generate parquet files.")

In [ ]:
# Show sample community reports
if not community_reports_df.empty:
    print("=== Sample Community Reports (first 2) ===")
    cols = [c for c in ["title", "summary", "rank", "level"] if c in community_reports_df.columns]
    for idx, row in community_reports_df[cols].head(2).iterrows():
        print(f"\n--- Community Report {idx} ---")
        for col in cols:
            val = str(row[col])[:200]
            print(f"{col}: {val}")
else:
    print("community_reports_df is empty -- run Section 6 first.")

## Section 8 — Prepare Documents for Qdrant

Build two document lists from GraphRAG outputs (with fallback to source JSON if parquet files are not yet available):
1. **entity_docs** -- one document per entity, enriched with relationship context
2. **community_docs** -- one document per community report summary

In [ ]:
import openai
from tqdm import tqdm

oai = openai.OpenAI(api_key=OPENAI_API_KEY)
EMBED_MODEL = "text-embedding-3-small"
EMBED_DIM   = 1536

def embed_texts(texts: list, batch_size: int = 100) -> list:
    """Embed a list of texts using OpenAI, returning list of float vectors."""
    all_embeddings = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Embedding"):
        batch = texts[i:i+batch_size]
        resp = oai.embeddings.create(model=EMBED_MODEL, input=batch)
        all_embeddings.extend([e.embedding for e in resp.data])
    return all_embeddings


# Build entity_docs
entity_docs = []

if not entities_df.empty:
    print("Building entity docs from GraphRAG parquet outputs...")
    # Build relationship lookup: entity title -> list of relationship descriptions
    rel_map = {}
    if not relationships_df.empty:
        src_col  = "source" if "source" in relationships_df.columns else None
        tgt_col  = "target" if "target" in relationships_df.columns else None
        desc_col = "description" if "description" in relationships_df.columns else None
        if src_col and tgt_col:
            for _, row in relationships_df.iterrows():
                src  = str(row[src_col])
                tgt  = str(row[tgt_col])
                desc = str(row[desc_col]) if desc_col else ""
                rel_map.setdefault(src, []).append(f"{tgt}: {desc}")
                rel_map.setdefault(tgt, []).append(f"{src}: {desc}")

    for _, row in entities_df.iterrows():
        title = str(row.get("title", ""))
        etype = str(row.get("type", ""))
        desc  = str(row.get("description", ""))
        entity_id = str(row.get("id", title))

        related = rel_map.get(title, [])[:5]
        rel_text = "; ".join(related) if related else "no direct relationships recorded"

        text = (
            f"{title} [{etype}]: {desc} "
            f"Related entities: {rel_text}"
        )
        entity_docs.append({
            "id":    entity_id,
            "title": title,
            "type":  etype,
            "text":  text,
        })
    print(f"Built {len(entity_docs)} entity documents from GraphRAG outputs.")
else:
    print("entities_df is empty -- building fallback entity docs from source JSON.")
    for d in DRIVERS:
        text = (
            f"{d['name']} [driver]: Professional driver with rating {d['rating']}, "
            f"{d['trips']} trips completed, operating a {d['vehicle']} in the {d['zone']} zone. "
            f"Specializes in {d['specialty']}. Languages: {', '.join(d['languages'])}. "
            f"Active for {d['years_active']} years on the platform."
        )
        entity_docs.append({"id": d["id"], "title": d["name"], "type": "driver", "text": text})
    for r in RIDERS:
        text = (
            f"{r['name']} [rider]: {r['tier']}-tier rider with {r['trips']} trips, "
            f"rating {r['rating']}, home zone {r['home_zone']}. "
            f"Prefers {r['preferred_vehicle']} vehicles. Frequent route: {r['frequent_route']}."
        )
        entity_docs.append({"id": r["id"], "title": r["name"], "type": "rider", "text": text})
    for z in ZONES:
        text = (
            f"{z['name']} [zone]: Service zone with demand score {z['demand_score']}/10, "
            f"average wait {z['avg_wait_min']} minutes, {z['surge_freq']} surge frequency. "
            f"Peak hours: {z['peak_hours']}. Top destinations: {', '.join(z['top_destinations'])}."
        )
        entity_docs.append({"id": z["id"], "title": z["name"], "type": "zone", "text": text})
    print(f"Built {len(entity_docs)} fallback entity documents from source JSON.")


# Build community_docs
community_docs = []

if not community_reports_df.empty:
    print("\nBuilding community docs from GraphRAG parquet outputs...")
    for idx, row in community_reports_df.iterrows():
        title        = str(row.get("title",        f"Community {row.get('community', idx)}"))
        summary      = str(row.get("summary",      ""))
        full_content = str(row.get("full_content", summary))
        text         = f"{title}\n\n{full_content if full_content not in ('nan', '') else summary}"
        community_docs.append({
            "id":    str(row.get("id", row.get("community", str(idx)))),
            "title": title,
            "text":  text,
        })
    print(f"Built {len(community_docs)} community documents from GraphRAG outputs.")
else:
    print("\ncommunity_reports_df is empty -- creating fallback community documents.")
    community_docs = [
        {
            "id": "C001",
            "title": "Downtown Driver Community",
            "text": (
                "Downtown Driver Community\n\n"
                "The Downtown zone hosts two of the platform's top-rated drivers: Carlos Mendez (4.92 rating, "
                "1,843 trips, airport transfer specialist) and Ahmed Hassan (4.83 rating, 1,567 trips, family "
                "ride specialist). Together they serve the high-demand Downtown zone with frequent surge pricing "
                "during peak hours (7-9am, 5-8pm). Key riders include Gold-tier Sarah Johnson and Platinum-tier "
                "Lisa Chen. The Downtown-Airport corridor generates significant revenue with 1.2-1.3x surge "
                "multipliers. Carlos Mendez has earned repeat business from multiple premium customers."
            ),
        },
        {
            "id": "C002",
            "title": "Airport and Midtown Operations Community",
            "text": (
                "Airport and Midtown Operations Community\n\n"
                "The Airport zone achieves the shortest wait times on the platform (2.8 minutes average) "
                "with a high demand score of 8.5. James Okafor (D003) is the specialist driver for this zone, "
                "knowledgeable about terminal layouts and pickup procedures. Surge pricing is common during "
                "flight peaks (6-8am, 4-7pm). Marcus Williams (R004, Silver-tier) frequently uses this zone. "
                "Midtown (demand score 7.8) is served primarily by Priya Sharma (4.87 rating), who specializes "
                "in corporate rides. Tom Bradley (R002) is a regular Midtown to Suburbs customer. Priya's "
                "professional service has built a loyal corporate clientele."
            ),
        },
        {
            "id": "C003",
            "title": "Premium Suburban Luxury Rides Community",
            "text": (
                "Premium Suburban Luxury Rides Community\n\n"
                "Mei Lin (D004) operates the platform's only Tesla Model 3 from the Suburbs zone, achieving "
                "the highest rating (4.95) and most trips (2,341) among all drivers. Her eco-luxury service "
                "attracts Platinum-tier rider Lisa Chen (R003, perfect 5.0 rating, 678 trips) for Suburbs-Downtown "
                "routes. Despite lower base demand (score 5.3), the Suburbs zone generates high per-trip revenue "
                "with frequent 1.3-1.5x surge multipliers on longer routes. The premium Tesla positioning "
                "differentiates suburban service from commodity urban rides and attracts eco-conscious affluent riders."
            ),
        },
    ]
    print(f"Built {len(community_docs)} fallback community documents.")

In [ ]:
# Generate dense embeddings
print("Generating dense embeddings for entity docs...")
entity_texts = [doc["text"] for doc in entity_docs]
entity_embeddings = embed_texts(entity_texts)

print("\nGenerating dense embeddings for community docs...")
community_texts = [doc["text"] for doc in community_docs]
community_embeddings = embed_texts(community_texts)

print(f"\nEmbedding dimensions: {len(entity_embeddings[0])}")
print(f"Entity embeddings:    {len(entity_embeddings)}")
print(f"Community embeddings: {len(community_embeddings)}")

## Section 9 — Sparse Vectors with BM25

Build BM25 indexes for sparse vector search. The `rebuild_bm25` function supports live ingestion of new documents.

In [ ]:
from rank_bm25 import BM25Okapi
import numpy as np

def tokenize(text: str) -> list:
    """Simple whitespace tokenizer (lowercased)."""
    return text.lower().split()


def rebuild_bm25(texts: list) -> BM25Okapi:
    """Build (or rebuild) a BM25 index from a list of texts.
    Call this whenever new documents are added to support live ingestion."""
    corpus = [tokenize(t) for t in texts]
    return BM25Okapi(corpus)


def bm25_sparse_vector(bm25: BM25Okapi, query: str) -> dict:
    """Compute a sparse vector (indices + values) from BM25 scores for a query.
    Returns a dict with 'indices' and 'values' for Qdrant SparseVector.
    """
    tokens = tokenize(query)
    scores = bm25.get_scores(tokens)
    nonzero_mask = scores > 0
    indices = np.where(nonzero_mask)[0].tolist()
    values  = scores[nonzero_mask].tolist()
    if not indices:
        # Fallback: use top-1 to avoid empty sparse vector errors
        top_idx = int(np.argmax(scores))
        indices = [top_idx]
        values  = [float(scores[top_idx]) if scores[top_idx] > 0 else 1e-6]
    return {"indices": indices, "values": values}


# Build initial BM25 indexes
bm25_entities    = rebuild_bm25(entity_texts)
bm25_communities = rebuild_bm25(community_texts)

print(f"BM25 entity index built    : {len(entity_texts)} documents")
print(f"BM25 community index built : {len(community_texts)} documents")

# Verify sparse vector for a sample query
test_sv = bm25_sparse_vector(bm25_entities, "Tesla driver luxury rating")
print(f"\nSample sparse vector for 'Tesla driver luxury rating':")
print(f"  Non-zero indices: {len(test_sv['indices'])}")
print(f"  Top indices: {test_sv['indices'][:5]}")
print(f"  Top values:  {[round(v, 4) for v in test_sv['values'][:5]]}")

## Section 10 — Create Qdrant Collections

Two collections with named dense + sparse vectors:
- `uber_entities` -- for local (entity-level) GraphRAG search
- `uber_communities` -- for global (community-level) GraphRAG search

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, VectorParams, SparseVectorParams, SparseIndexParams,
    PointStruct, SparseVector, NamedVector, NamedSparseVector,
    Prefetch, FusionQuery, Fusion
)

qdrant = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)

COLLECTION_ENTITIES    = "uber_entities"
COLLECTION_COMMUNITIES = "uber_communities"

def create_hybrid_collection(name: str):
    """Create a Qdrant collection with named dense + sparse vectors."""
    if qdrant.collection_exists(name):
        qdrant.delete_collection(name)
        print(f"  Deleted existing collection: {name}")

    qdrant.create_collection(
        collection_name=name,
        vectors_config={
            "dense": VectorParams(size=EMBED_DIM, distance=Distance.COSINE)
        },
        sparse_vectors_config={
            "sparse": SparseVectorParams(
                index=SparseIndexParams(on_disk=False)
            )
        }
    )
    print(f"  Created collection: {name}")

print("Creating Qdrant collections...")
create_hybrid_collection(COLLECTION_ENTITIES)
create_hybrid_collection(COLLECTION_COMMUNITIES)

print("\nCollections created successfully.")

In [ ]:
def make_point_id(str_id: str) -> int:
    """Convert a string ID to a positive int Qdrant point ID."""
    return hash(str_id) % (2**31)


def build_points(docs, embeddings, bm25_index):
    """Build PointStruct list from docs + dense embeddings + BM25 sparse vectors."""
    points = []
    for doc, dense_vec in zip(docs, embeddings):
        sparse_vec = bm25_sparse_vector(bm25_index, doc["text"])
        point = PointStruct(
            id=make_point_id(doc["id"]),
            vector={
                "dense":  dense_vec,
                "sparse": SparseVector(
                    indices=sparse_vec["indices"],
                    values=sparse_vec["values"]
                )
            },
            payload={
                "original_id": doc["id"],
                "title":       doc.get("title", ""),
                "type":        doc.get("type",  ""),
                "text":        doc["text"],
            }
        )
        points.append(point)
    return points


# Upload entity points
print("Uploading entity documents to Qdrant...")
entity_points = build_points(entity_docs, entity_embeddings, bm25_entities)
qdrant.upsert(collection_name=COLLECTION_ENTITIES, points=entity_points)
print(f"  Upserted {len(entity_points)} entity points")

# Upload community points
print("\nUploading community documents to Qdrant...")
community_points = build_points(community_docs, community_embeddings, bm25_communities)
qdrant.upsert(collection_name=COLLECTION_COMMUNITIES, points=community_points)
print(f"  Upserted {len(community_points)} community points")

print("\nAll data uploaded to Qdrant.")
print(f"  {COLLECTION_ENTITIES}:    {qdrant.count(COLLECTION_ENTITIES).count} points")
print(f"  {COLLECTION_COMMUNITIES}: {qdrant.count(COLLECTION_COMMUNITIES).count} points")

## Section 11 — CRUD Operations on Qdrant

Demonstrate Create, Read, Update, and Delete operations on the `uber_entities` collection.

In [ ]:
# CREATE -- initial upsert already done in Section 10
print("=" * 60)
print("CREATE -- collection already populated in Section 10")
info = qdrant.get_collection(COLLECTION_ENTITIES)
print(f"  Collection '{COLLECTION_ENTITIES}' has {info.points_count} points")

# READ -- retrieve a specific driver entity
print("\n" + "=" * 60)
print("READ -- fetch driver Carlos Mendez by ID")

# Find Carlos Mendez in entity_docs
carlos_doc = next(
    (doc for doc in entity_docs if "carlos" in doc["title"].lower() or doc["id"] == "D001"),
    entity_docs[0]  # fallback to first doc
)
carlos_point_id = make_point_id(carlos_doc["id"])

results = qdrant.retrieve(
    collection_name=COLLECTION_ENTITIES,
    ids=[carlos_point_id],
    with_payload=True,
    with_vectors=False
)
if results:
    r = results[0]
    print(f"  Found point ID: {r.id}")
    print(f"  Title: {r.payload.get('title', 'N/A')}")
    print(f"  Type:  {r.payload.get('type', 'N/A')}")
    print(f"  Text (snippet): {r.payload.get('text', '')[:200]}...")
else:
    print(f"  Point {carlos_point_id} not found.")
print("\nREAD complete.")

In [ ]:
# UPDATE -- simulate Carlos Mendez receiving a rating boost
print("=" * 60)
print("UPDATE -- Carlos Mendez rating updated from 4.92 to 4.97 after 50 new trips")

updated_text = (
    "Carlos Mendez [driver]: UPDATED profile -- Professional driver with rating 4.97 "
    "(was 4.92), now 1893 total trips completed, operating a Toyota Camry 2021 in the Downtown zone. "
    "Specializes in airport transfers. Languages: English, Spanish. "
    "Recently earned Platinum Driver status after consistent 5-star performance. "
    "Related entities: Sarah Johnson: frequent rider on Downtown-Airport route; "
    "Lisa Chen: repeat Platinum-tier customer who gave 5-star reviews."
)

# Re-embed updated text
updated_embedding = oai.embeddings.create(model=EMBED_MODEL, input=[updated_text]).data[0].embedding

# Rebuild BM25 with updated doc text
updated_entity_texts = [
    updated_text if ("carlos" in doc["title"].lower() or doc["id"] == "D001") else doc["text"]
    for doc in entity_docs
]
bm25_entities_v2 = rebuild_bm25(updated_entity_texts)
updated_sparse = bm25_sparse_vector(bm25_entities_v2, updated_text)

qdrant.upsert(
    collection_name=COLLECTION_ENTITIES,
    points=[PointStruct(
        id=carlos_point_id,
        vector={
            "dense":  updated_embedding,
            "sparse": SparseVector(
                indices=updated_sparse["indices"],
                values=updated_sparse["values"]
            )
        },
        payload={
            "original_id": carlos_doc["id"],
            "title":       "Carlos Mendez",
            "type":        "driver",
            "text":        updated_text,
            "rating":      4.97,
            "trips":       1893,
            "status":      "platinum_driver",
        }
    )]
)
print("  Updated Carlos Mendez point (dense + sparse vectors + payload refreshed)")

verify = qdrant.retrieve(collection_name=COLLECTION_ENTITIES, ids=[carlos_point_id], with_payload=True)
if verify:
    print(f"  Verified -- new rating in payload: {verify[0].payload.get('rating', 'N/A')}")
    print(f"  Status: {verify[0].payload.get('status', 'N/A')}")
print("UPDATE complete.")

In [ ]:
# DELETE -- remove a deactivated entity
print("=" * 60)
print("DELETE -- remove a deactivated test entity")

# Insert a temporary entity to delete
temp_text = "Deactivated Driver Test [driver]: This account has been suspended due to policy violation."
temp_embedding = oai.embeddings.create(model=EMBED_MODEL, input=[temp_text]).data[0].embedding
temp_sparse    = bm25_sparse_vector(bm25_entities, temp_text)
temp_id        = make_point_id("DEACTIVATED_TEST_001")

qdrant.upsert(
    collection_name=COLLECTION_ENTITIES,
    points=[PointStruct(
        id=temp_id,
        vector={
            "dense":  temp_embedding,
            "sparse": SparseVector(indices=temp_sparse["indices"], values=temp_sparse["values"])
        },
        payload={"original_id": "DEACTIVATED_TEST_001", "title": "Deactivated Driver Test",
                 "type": "driver", "text": temp_text, "status": "deactivated"}
    )]
)
count_before = qdrant.count(COLLECTION_ENTITIES).count
print(f"  Inserted deactivated driver (point ID: {temp_id})")
print(f"  Collection size before delete: {count_before}")

qdrant.delete(
    collection_name=COLLECTION_ENTITIES,
    points_selector=[temp_id]
)

count_after = qdrant.count(COLLECTION_ENTITIES).count
print(f"  Deleted point {temp_id}")
print(f"  Collection size after delete:  {count_after}")
print("DELETE complete.")

## Section 12 — Dense, Sparse, and Hybrid RRF Search

In [ ]:
def dense_search(query: str, collection: str, top_k: int = 3) -> list:
    """Search using dense (cosine similarity) vectors only."""
    q_vec = oai.embeddings.create(model=EMBED_MODEL, input=[query]).data[0].embedding
    results = qdrant.search(
        collection_name=collection,
        query_vector=NamedVector(name="dense", vector=q_vec),
        limit=top_k,
        with_payload=True
    )
    return results


def sparse_search(query: str, collection: str, bm25_index, top_k: int = 3) -> list:
    """Search using sparse (BM25) vectors only."""
    sv = bm25_sparse_vector(bm25_index, query)
    results = qdrant.search(
        collection_name=collection,
        query_vector=NamedSparseVector(
            name="sparse",
            vector=SparseVector(indices=sv["indices"], values=sv["values"])
        ),
        limit=top_k,
        with_payload=True
    )
    return results


def hybrid_rrf_search(query: str, collection: str, bm25_index, top_k: int = 3) -> list:
    """Hybrid search using Reciprocal Rank Fusion (RRF) of dense + sparse."""
    q_vec = oai.embeddings.create(model=EMBED_MODEL, input=[query]).data[0].embedding
    sv = bm25_sparse_vector(bm25_index, query)

    results = qdrant.query_points(
        collection_name=collection,
        prefetch=[
            Prefetch(
                query=NamedVector(name="dense", vector=q_vec),
                limit=top_k * 3
            ),
            Prefetch(
                query=NamedSparseVector(
                    name="sparse",
                    vector=SparseVector(indices=sv["indices"], values=sv["values"])
                ),
                limit=top_k * 3
            ),
        ],
        query=FusionQuery(fusion=Fusion.RRF),
        limit=top_k,
        with_payload=True
    )
    return results.points


def print_results(results, label: str):
    print(f"\n{label}:")
    for i, r in enumerate(results, 1):
        payload = r.payload if hasattr(r, 'payload') else {}
        title   = payload.get('title', 'N/A')
        etype   = payload.get('type',  'N/A')
        score   = round(r.score, 4) if hasattr(r, 'score') else 'N/A'
        text    = payload.get('text', '')[:100]
        print(f"  {i}. [{score}] {title} ({etype}) -- {text}...")


# Run all three search types
query = "highest rated driver with luxury electric vehicle"
print(f"Query: '{query}'\n")

dense_results  = dense_search(query, COLLECTION_ENTITIES)
sparse_results = sparse_search(query, COLLECTION_ENTITIES, bm25_entities)
hybrid_results = hybrid_rrf_search(query, COLLECTION_ENTITIES, bm25_entities)

print_results(dense_results,  "DENSE search results")
print_results(sparse_results, "SPARSE (BM25) search results")
print_results(hybrid_results, "HYBRID RRF search results")

## Section 13 — Local GraphRAG Search

Retrieve entities from `uber_entities`, enrich with relationship context from `relationships_df`, then use GPT-4o-mini to synthesize an answer. This is local GraphRAG: specific entity-level queries.

In [ ]:
def local_graphrag_search(query: str, top_k: int = 5) -> str:
    """Local GraphRAG search: hybrid entity retrieval + relationship enrichment + LLM synthesis."""
    print(f"Local GraphRAG query: '{query}'")

    # 1. Retrieve top-k entities via hybrid search
    raw_results = hybrid_rrf_search(query, COLLECTION_ENTITIES, bm25_entities, top_k=top_k)

    # 2. Build enriched context: entity text + relationships from DataFrame
    context_blocks = []
    for hit in raw_results:
        payload = hit.payload
        title   = payload.get("title", "")
        text    = payload.get("text", "")

        # Enrich with relationship data from graphrag output (if available)
        rel_context = ""
        if not relationships_df.empty:
            src_col  = "source" if "source" in relationships_df.columns else None
            tgt_col  = "target" if "target" in relationships_df.columns else None
            desc_col = "description" if "description" in relationships_df.columns else None
            if src_col and tgt_col:
                related = relationships_df[
                    (relationships_df[src_col].str.upper() == title.upper()) |
                    (relationships_df[tgt_col].str.upper() == title.upper())
                ]
                if not related.empty and desc_col:
                    rel_texts = related[desc_col].dropna().head(3).tolist()
                    rel_context = "\n  Relationships: " + "; ".join(str(r) for r in rel_texts)

        context_blocks.append(f"Entity: {title}\n{text}{rel_context}")

    context = "\n\n---\n\n".join(context_blocks)

    # 3. LLM synthesis
    system_prompt = (
        "You are a knowledgeable assistant for the Uber marketplace platform. "
        "Use the provided entity context to answer the question accurately. "
        "Cite specific entities, ratings, and facts from the context."
    )
    user_prompt = f"Context:\n{context}\n\nQuestion: {query}\n\nAnswer:"

    response = oai.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt}
        ],
        max_tokens=400,
        temperature=0.1
    )
    return response.choices[0].message.content


# Example 1
q1 = "Which driver is best for airport transfers and why?"
answer1 = local_graphrag_search(q1)
print(f"\nQ: {q1}")
print(f"A: {answer1}")

In [ ]:
# Example 2
q2 = "What are the relationships between Platinum-tier riders and top-rated drivers?"
answer2 = local_graphrag_search(q2)
print(f"Q: {q2}")
print(f"A: {answer2}")

## Section 14 — Global GraphRAG Search

Query `uber_communities` to answer high-level aggregated questions using community report summaries as context. This is global GraphRAG: trend and pattern queries that local search misses.

In [ ]:
def global_graphrag_search(query: str, top_k: int = 3) -> str:
    """Global GraphRAG search: community report retrieval + LLM synthesis."""
    print(f"Global GraphRAG query: '{query}'")

    # 1. Retrieve top-k community reports via hybrid search
    raw_results = hybrid_rrf_search(query, COLLECTION_COMMUNITIES, bm25_communities, top_k=top_k)

    # 2. Assemble community context
    community_contexts = []
    for hit in raw_results:
        payload = hit.payload
        title   = payload.get("title", "Community Report")
        text    = payload.get("text", "")
        community_contexts.append(f"[{title}]\n{text}")

    context = "\n\n===\n\n".join(community_contexts)

    # 3. LLM synthesis with community-level context
    system_prompt = (
        "You are a strategic analyst for the Uber marketplace platform. "
        "Use the provided community-level summaries to answer high-level, "
        "aggregated questions about the platform's patterns, trends, and performance. "
        "Synthesize insights across multiple communities when relevant."
    )
    user_prompt = f"Community Reports:\n{context}\n\nQuestion: {query}\n\nAnswer:"

    response = oai.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt}
        ],
        max_tokens=500,
        temperature=0.1
    )
    return response.choices[0].message.content


# Example 1
q3 = "What are the overall service quality trends and premium market segments across all zones?"
answer3 = global_graphrag_search(q3)
print(f"\nQ: {q3}")
print(f"A: {answer3}")

In [ ]:
# Example 2
q4 = "Which zones have the highest revenue potential and how should drivers be allocated?"
answer4 = global_graphrag_search(q4)
print(f"Q: {q4}")
print(f"A: {answer4}")

## Section 15 — Live Ingestion Demo

Add a new driver record, call `rebuild_bm25`, generate embeddings, and upsert to Qdrant -- demonstrating real-time knowledge base extension without full re-indexing.

In [ ]:
print("=" * 60)
print("LIVE INGESTION: Adding new driver Fatima Al-Zahra")
print("=" * 60)

new_driver = {
    "id": "D006",
    "name": "Fatima Al-Zahra",
    "vehicle": "BMW 5 Series 2024",
    "rating": 4.98,
    "trips": 432,
    "zone": "Midtown",
    "license": "FA-1124",
    "years_active": 1,
    "languages": ["English", "Arabic", "French"],
    "specialty": "Executive rides"
}

new_driver_text = (
    f"{new_driver['name']} [driver]: A newly joined driver with the highest current rating of "
    f"{new_driver['rating']} stars after {new_driver['trips']} trips. She operates a premium "
    f"{new_driver['vehicle']} (license {new_driver['license']}) in the {new_driver['zone']} zone, "
    f"specializing in {new_driver['specialty']}. Trilingual in "
    f"{', '.join(new_driver['languages'])}, Fatima serves high-profile corporate executives and "
    f"diplomatic clients. Despite only 1 year on the platform, her immaculate vehicle maintenance, "
    f"discretion, and five-star service have made her the go-to choice for premium Midtown rides. "
    f"Related entities: Midtown zone: primary service area with corporate clientele; "
    f"Priya Sharma: fellow Midtown specialist driver."
)

# Step 1: Rebuild BM25 with new document added
all_entity_texts_updated = entity_texts + [new_driver_text]
bm25_entities = rebuild_bm25(all_entity_texts_updated)  # update global index
print(f"Step 1: BM25 rebuilt with {len(all_entity_texts_updated)} documents")

# Step 2: Generate dense embedding
new_embedding = oai.embeddings.create(model=EMBED_MODEL, input=[new_driver_text]).data[0].embedding
print(f"Step 2: Dense embedding generated (dim={len(new_embedding)})")

# Step 3: Generate sparse vector using updated BM25 index
new_sparse = bm25_sparse_vector(bm25_entities, new_driver_text)
print(f"Step 3: Sparse vector computed ({len(new_sparse['indices'])} non-zero indices)")

# Step 4: Upsert to Qdrant
new_point_id = make_point_id(new_driver["id"])
qdrant.upsert(
    collection_name=COLLECTION_ENTITIES,
    points=[PointStruct(
        id=new_point_id,
        vector={
            "dense":  new_embedding,
            "sparse": SparseVector(indices=new_sparse["indices"], values=new_sparse["values"])
        },
        payload={
            "original_id": new_driver["id"],
            "title":       new_driver["name"],
            "type":        "driver",
            "text":        new_driver_text,
            "rating":      new_driver["rating"],
            "zone":        new_driver["zone"],
            "specialty":   new_driver["specialty"],
            "status":      "active",
        }
    )]
)
print(f"Step 4: Upserted to Qdrant (point ID: {new_point_id})")
print(f"        Collection now has {qdrant.count(COLLECTION_ENTITIES).count} points")

# Step 5: Verify retrieval
print("\nStep 5: Verifying retrieval via hybrid search...")
verify_query = "executive rides BMW premium Midtown driver"
verify_results = hybrid_rrf_search(verify_query, COLLECTION_ENTITIES, bm25_entities, top_k=3)

print(f"Query: '{verify_query}'")
found = False
for r in verify_results:
    title = r.payload.get("title", "")
    print(f"  Result: {title} -- {r.payload.get('text', '')[:80]}...")
    if "fatima" in title.lower():
        found = True
        print("  *** Fatima Al-Zahra successfully retrieved! ***")

if not found:
    # Try direct retrieval as confirmation
    direct = qdrant.retrieve(collection_name=COLLECTION_ENTITIES, ids=[new_point_id], with_payload=True)
    if direct:
        print(f"  Direct retrieval confirmed: {direct[0].payload.get('title')} (point ID: {new_point_id})")

print("\nLive ingestion demo complete.")

## Section 16 — Search Comparison Table

Side-by-side comparison of all five search strategies on the same query.

In [ ]:
import pandas as pd

comparison_query = "best rated driver for premium luxury rides"
print(f"Comparison query: '{comparison_query}'\n")

# Run all search strategies
dense_r   = dense_search(comparison_query, COLLECTION_ENTITIES, top_k=3)
sparse_r  = sparse_search(comparison_query, COLLECTION_ENTITIES, bm25_entities, top_k=3)
hybrid_r  = hybrid_rrf_search(comparison_query, COLLECTION_ENTITIES, bm25_entities, top_k=3)

local_answer  = local_graphrag_search(comparison_query, top_k=4)
global_answer = global_graphrag_search(comparison_query, top_k=3)

rows = []

for i, r in enumerate(dense_r, 1):
    rows.append({
        "Strategy":   "Dense (cosine)",
        "Rank":       i,
        "Result":     r.payload.get("title", "N/A"),
        "Type":       r.payload.get("type",  "N/A"),
        "Score":      round(r.score, 4),
        "Snippet":    r.payload.get("text", "")[:80] + "..."
    })

for i, r in enumerate(sparse_r, 1):
    rows.append({
        "Strategy":   "Sparse (BM25)",
        "Rank":       i,
        "Result":     r.payload.get("title", "N/A"),
        "Type":       r.payload.get("type",  "N/A"),
        "Score":      round(r.score, 4),
        "Snippet":    r.payload.get("text", "")[:80] + "..."
    })

for i, r in enumerate(hybrid_r, 1):
    rows.append({
        "Strategy":   "Hybrid RRF",
        "Rank":       i,
        "Result":     r.payload.get("title", "N/A"),
        "Type":       r.payload.get("type",  "N/A"),
        "Score":      round(r.score, 4) if hasattr(r, 'score') else "N/A",
        "Snippet":    r.payload.get("text", "")[:80] + "..."
    })

rows.append({
    "Strategy":   "Local GraphRAG",
    "Rank":       1,
    "Result":     "LLM-synthesized answer",
    "Type":       "entities+relationships",
    "Score":      "N/A",
    "Snippet":    local_answer[:120] + "..."
})
rows.append({
    "Strategy":   "Global GraphRAG",
    "Rank":       1,
    "Result":     "LLM-synthesized answer",
    "Type":       "community_reports",
    "Score":      "N/A",
    "Snippet":    global_answer[:120] + "..."
})

comparison_df = pd.DataFrame(rows)
pd.set_option("display.max_colwidth", 100)
pd.set_option("display.width", 220)
print(comparison_df.to_string(index=False))

## Section 17 — Summary

### Architecture Components

| Component | Technology | Role |
|-----------|-----------|------|
| Knowledge Graph Construction | Microsoft GraphRAG (`graphrag>=2.0.0`) | Extracts entities, relationships, communities from text via LLM |
| LLM for Extraction | OpenAI `gpt-4o-mini` | Entity/relationship extraction and community report generation |
| Graph Outputs | Parquet files (entities, relationships, community_reports, text_units) | Structured knowledge base on disk |
| Dense Vectors | OpenAI `text-embedding-3-small` (1536-dim) | Semantic similarity search |
| Sparse Vectors | BM25 (rank-bm25) with `rebuild_bm25` | Keyword/lexical search, supports live ingestion |
| Vector Database | Qdrant Serverless | Hybrid search with named vectors, CRUD operations |
| Local Search Collection | `uber_entities` | Entity + relationship context for specific queries |
| Global Search Collection | `uber_communities` | Community report context for aggregated queries |
| Hybrid Fusion | Reciprocal Rank Fusion (RRF) | Combines dense + sparse results without score normalization |
| Local GraphRAG | Hybrid entity search + relationship enrichment + GPT-4o-mini | Specific entity-level answers |
| Global GraphRAG | Community report search + GPT-4o-mini | High-level trend and pattern answers |

### 5 Production Patterns

1. **Incremental Re-indexing**: Use `graphrag index --update` with new input files for incremental updates; call `rebuild_bm25` and upsert only changed entities to Qdrant without full re-indexing.

2. **Tiered Query Routing**: Route queries through a classifier -- specific entity questions go to Local GraphRAG (entity + relationship context), while aggregate/trend questions go to Global GraphRAG (community report context). This matches the GraphRAG paper's local vs. global search distinction.

3. **Entity Deduplication**: GraphRAG may extract the same real-world entity with slight name variations. Use entity resolution (fuzzy string matching or embedding cosine similarity) to merge duplicates before upserting to Qdrant.

4. **Community Hierarchy Levels**: GraphRAG generates communities at multiple resolution levels (level 0 = fine-grained, level 2+ = broad clusters). Store the `level` field in Qdrant payloads and filter by level to control answer granularity based on query type.

5. **Payload Filtering for Freshness**: Store `last_updated` timestamps in Qdrant payloads. Use payload filters in queries to exclude stale entries, ensuring search results always reflect the most current data without requiring full re-indexing.